<a href="https://colab.research.google.com/github/fawaztanigbola/Nigeria-ForeCasting-Model/blob/main/Chronologically_Splitting_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  Chronological Dataset Splitting

### Objective
Divide the prepared time-series data into training, validation and testing periods while preserving chronological order.

### Splitting Strategy
A chronological 70% / 15% / 15% split was implemented:

- **Training (70%):** Used to fit the model parameters.
- **Validation (15%):** Used to monitor model performance and guide model selection.
- **Testing (15%):** Reserved for final evaluation on unseen future observations.

### Method
1. Loaded the prepared daily temperature dataset.
2. Used the target date of each valid seven-day sequence to determine its assigned period.
3. Assigned sequences with target dates before the 70% boundary to training.
4. Assigned sequences between the 70% and 85% boundaries to validation.
5. Assigned all remaining sequences to testing.
6. Verified that the three periods were chronologically ordered and non-overlapping.

### Leakage Prevention
Random splitting was avoided because this is a time-series forecasting problem. Randomly distributing sequences could allow future target observations to enter the training set.

The split is based on prediction target dates. Historical input windows may include observations from the preceding period because those observations would already be available when forecasting the target date.

### Model Comparison
The baseline LSTM and geospatial LSTM use identical sequence indexes and chronological boundaries. This ensures a fair comparison of their forecasting performance.

### Output
The split indexes are saved in:

`model_data/chronological_splits.npz`

The file contains:
- Training sequence indexes
- Validation sequence indexes
- Testing sequence indexes
- Training and validation boundary indexes

### Expected Outcome
Three chronological datasets are available for model development and evaluation without random time-series splitting.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

project_dir = Path(
    "/content/drive/MyDrive/Nigeria_Temperature_Project"
)

data_dir = project_dir / "model_data"

data = np.load(
    data_dir / "prepared_model_data.npz"
)

dates = pd.to_datetime(data["dates"])

print("First date:", dates.min())
print("Last date:", dates.max())
print("Total days:", len(dates))

Mounted at /content/drive
First date: 2015-01-01 00:00:00
Last date: 2024-12-31 00:00:00
Total days: 3653


In [2]:
total_days = len(dates)

train_end = int(total_days * 0.70)
val_end = int(total_days * 0.85)

print("Training ends:", dates[train_end - 1])
print("Validation ends:", dates[val_end - 1])
print("Testing begins:", dates[val_end])

Training ends: 2021-12-31 00:00:00
Validation ends: 2023-07-02 00:00:00
Testing begins: 2023-07-03 00:00:00


In [3]:
target_times = data["train_pairs"][:, 0] if False else None

In [4]:
temperature = data["temperature"]

# A day is valid when both Tmin and Tmax exist
valid_day = np.isfinite(temperature).all(axis=2)

# Find valid 8-day windows:
# 7 input days + 1 prediction target
valid_windows = np.lib.stride_tricks.sliding_window_view(
    valid_day,
    window_shape=8,
    axis=0
).all(axis=-1)

# Identify target dates and locations
window_starts, location_indexes = np.where(valid_windows)

target_times = window_starts + 7

# Store all valid sequences
all_pairs = np.column_stack([
    target_times,
    location_indexes
]).astype("int32")

In [5]:
train_pairs = all_pairs[
    all_pairs[:, 0] < train_end
]

val_pairs = all_pairs[
    (all_pairs[:, 0] >= train_end) &
    (all_pairs[:, 0] < val_end)
]

test_pairs = all_pairs[
    all_pairs[:, 0] >= val_end
]

print("Training sequences:", len(train_pairs))
print("Validation sequences:", len(val_pairs))
print("Testing sequences:", len(test_pairs))

Training sequences: 30814200
Validation sequences: 6622032
Testing sequences: 6622032


In [6]:
train_dates = dates[train_pairs[:, 0]]
val_dates = dates[val_pairs[:, 0]]
test_dates = dates[test_pairs[:, 0]]

print("TRAINING")
print(train_dates.min(), "to", train_dates.max())

print("\nVALIDATION")
print(val_dates.min(), "to", val_dates.max())

print("\nTESTING")
print(test_dates.min(), "to", test_dates.max())

# Confirm chronological order
assert train_dates.max() < val_dates.min()
assert val_dates.max() < test_dates.min()

print("\nChronological split verified.")

TRAINING
2015-01-08 00:00:00 to 2021-12-31 00:00:00

VALIDATION
2022-01-01 00:00:00 to 2023-07-02 00:00:00

TESTING
2023-07-03 00:00:00 to 2024-12-31 00:00:00

Chronological split verified.


In [7]:
np.savez_compressed(
    data_dir / "chronological_splits.npz",

    train_pairs=train_pairs,
    val_pairs=val_pairs,
    test_pairs=test_pairs,

    train_end=train_end,
    val_end=val_end
)

print("Chronological splits saved successfully.")

Chronological splits saved successfully.


In [8]:
# Baseline model
baseline_train_pairs = train_pairs
baseline_val_pairs = val_pairs
baseline_test_pairs = test_pairs

# Geospatial model
geo_train_pairs = train_pairs
geo_val_pairs = val_pairs
geo_test_pairs = test_pairs

assert np.array_equal(
    baseline_train_pairs,
    geo_train_pairs
)

assert np.array_equal(
    baseline_val_pairs,
    geo_val_pairs
)

assert np.array_equal(
    baseline_test_pairs,
    geo_test_pairs
)

print("Both models use identical chronological splits.")

Both models use identical chronological splits.
